# Foodbank Agent — Evaluation Notebook

Three sections:
1. **Sheet import** — load both Google Sheets, inspect shape and quality
2. **LLM connectivity** — one quick call to confirm the API key and model are reachable
3. **Evaluations** — 4 queries where we can independently verify the correct answer with pandas

In [ ]:
import os
import re
import sys

import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # picks up .env in the project root
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

from agent import run_query
from sheets import fetch_logins, fetch_registrations

PROVIDER = "Groq"  # change to "Cerebras" to test the other provider

---
## 1. Load Data from Google Sheets

In [ ]:
print("Loading registrations...")
registrations = pd.DataFrame(fetch_registrations())
print(f"  {len(registrations):,} rows × {len(registrations.columns)} columns")

print("Loading logins...")
logins = pd.DataFrame(fetch_logins())
print(f"  {len(logins):,} rows × {len(logins.columns)} columns")

In [ ]:
print("Registrations columns:", list(registrations.columns))
registrations.head(3)

In [ ]:
print("Logins columns:", list(logins.columns))
logins.head(3)

In [ ]:
# Basic data-quality summary
print("=== Registrations ===")
print(registrations[["Sex", "Dietary Requirements"]].apply(lambda s: s.value_counts().head(5)))
print("\n=== Logins by Day ===")
print(logins["Day"].value_counts())

---
## 2. LLM Connectivity Check

A single trivial call — if this errors the provider key or model is wrong.

In [ ]:
probe = run_query("How many rows are in the registrations sheet?", registrations, logins, provider=PROVIDER)

if probe.error:
    print("FAIL — agent returned an error:")
    print(probe.error)
else:
    print(f"OK  — {PROVIDER} is reachable")
    print("Response:", probe.text)

---
## 3. Evaluations

Each cell:
1. Computes the **ground-truth answer** directly with pandas
2. Asks the **LLM** the same question via `run_query`
3. Checks whether the expected number appears in the LLM's response text

Helper:

In [ ]:
def evaluate(question: str, expected: int, provider: str = PROVIDER) -> bool:
    """Run a query and check the LLM's answer contains the expected number."""
    result = run_query(question, registrations, logins, provider=provider)
    if result.error:
        print(f"  FAIL (agent error): {result.error[:120]}")
        return False
    # Accept the number appearing anywhere in the response text
    found = str(expected) in re.sub(r"[,_]", "", result.text)
    status = "PASS" if found else "FAIL"
    print(f"  {status}")
    print(f"  Expected : {expected:,}")
    print(f"  LLM said : {result.text.strip()}")
    return found

### Eval 1 — Total registered users

In [ ]:
expected = len(registrations)
print(f"Ground truth: {expected:,} registered users")
evaluate("How many people are registered in total?", expected)

### Eval 2 — Halal dietary requirements

In [ ]:
expected = registrations["Dietary Requirements"].str.contains("Halal", case=False, na=False).sum()
print(f"Ground truth: {expected:,} people with Halal requirements")
evaluate("How many people have Halal dietary requirements?", expected)

### Eval 3 — Total login visits

In [ ]:
expected = len(logins)
print(f"Ground truth: {expected:,} total login visits")
evaluate("How many total visits have been logged?", expected)

### Eval 4 — Male users

In [ ]:
expected = registrations["Sex"].str.strip().str.lower().eq("male").sum()
print(f"Ground truth: {expected:,} male users")
evaluate("How many male users are registered?", expected)

---
## Summary

In [ ]:
evals = [
    ("Total registrations",    len(registrations),                                                             "How many people are registered in total?"),
    ("Halal requirements",     int(registrations["Dietary Requirements"].str.contains("Halal", case=False, na=False).sum()), "How many people have Halal dietary requirements?"),
    ("Total logins",           len(logins),                                                                    "How many total visits have been logged?"),
    ("Male users",             int(registrations["Sex"].str.strip().str.lower().eq("male").sum()),             "How many male users are registered?"),
]

results = []
for name, expected, question in evals:
    print(f"\n--- {name} ---")
    passed = evaluate(question, expected)
    results.append((name, expected, passed))

print("\n" + "="*40)
print("EVALUATION SUMMARY")
print("="*40)
for name, expected, passed in results:
    icon = "✓" if passed else "✗"
    print(f"  {icon}  {name:25s}  (expected {expected:,})")
total = sum(p for _, _, p in results)
print(f"\n  {total}/{len(results)} passed")